[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DangHuuLong/Ai-Recruiter-Mini-Ai-Service/blob/experiment/cross-encoder-v0.4/notebooks/fine_tune_cross_encoder_v0_4_colab.ipynb)

# Fine-tune Cross-Encoder CV-JD v0.4

Fine-tune `cross-encoder/ms-marco-MiniLM-L-12-v2` với **5-class CrossEntropy loss** trên 4900 CV-JD pairs.
v0.4 chuyển từ regression sang direct classification để tối ưu label accuracy trực tiếp.

| | |
|---|---|
| **Dataset** | v0.3 — 4900 train / 1050 validation / 1050 test |
| **Base model** | `cross-encoder/ms-marco-MiniLM-L-12-v2` |
| **Loss** | `ClassificationLoss`: 5-class CrossEntropy (poor/weak/moderate/strong/excellent) |
| **Inference score** | softmax-weighted class centers: [20, 50, 67.5, 82.5, 95] |
| **max_length** | 512 tokens |
| **Branch** | `experiment/cross-encoder-v0.4` |

**Mục tiêu**: vượt LabelAcc 60.76% của v0.2 bằng cách optimize CE loss trực tiếp trên 5 bucket thay vì MSE.

In [ ]:
import os
if not os.path.exists('/content/Ai-Recruiter-Mini-Ai-Service'):
    !git clone https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service /content/Ai-Recruiter-Mini-Ai-Service

%cd /content/Ai-Recruiter-Mini-Ai-Service
!git checkout experiment/cross-encoder-v0.4
!git pull origin experiment/cross-encoder-v0.4

In [ ]:
!pip install -r requirements.txt

In [ ]:
from pathlib import Path
import json

data_dir = Path("datasets/versions/v0.3/cross_encoder")
for split in ("train", "validation", "test"):
    path = data_dir / f"cross_encoder_{split}.jsonl"
    lines = path.read_text(encoding="utf-8").strip().splitlines()
    first = json.loads(lines[0])
    print(f"{split:<12}: {len(lines):>5} pairs  | keys: {list(first.keys())}")

## Debug run — sanity check ClassificationLoss (1 epoch, 40 samples)

In [ ]:
!WANDB_MODE=disabled python -m training.fine_tune_cross_encoder \
    --loss classification \
    --evaluator label_acc \
    --base-model cross-encoder/ms-marco-MiniLM-L-12-v2 \
    --epochs 1 \
    --batch-size 4 \
    --max-train-samples 40 \
    --max-eval-samples 20

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Full training — 10 epochs, save to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

drive_base = "/content/drive/MyDrive/ai-recruiter"

import os, time
os.makedirs(f"{drive_base}/models/cross-encoder-cv-jd-v0.4", exist_ok=True)
time.sleep(3)

!WANDB_MODE=disabled python -m training.fine_tune_cross_encoder \
    --loss classification \
    --evaluator label_acc \
    --base-model cross-encoder/ms-marco-MiniLM-L-12-v2 \
    --output-dir {drive_base}/models/cross-encoder-cv-jd-v0.4 \
    --report-path artifacts/reports/fine_tune_cross_encoder_v0.4_report.json \
    --epochs 10 \
    --batch-size 16

In [ ]:
import json
from pathlib import Path

report = json.loads(
    Path("artifacts/reports/fine_tune_cross_encoder_v0.4_report.json").read_text(encoding="utf-8")
)
print(f"Base model : {report['base_model']}")
print(f"Loss       : {report['loss']}")
print()
print(json.dumps(report["metrics"], indent=2))

In [ ]:
import shutil
from pathlib import Path

reports_dir = Path(drive_base) / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(
    "artifacts/reports/fine_tune_cross_encoder_v0.4_report.json",
    reports_dir / "fine_tune_cross_encoder_v0.4_report.json",
)
print(f"Saved to {reports_dir}")